In [5]:
#============================================================
#Complex PDF Parser for RAG
#Text + Tables + Images + OCR + LangChain Documents
#============================================================

In [6]:
# Install dependencies

import os
import json
import fitz  # PyMuPDF
import pdfplumber
import pandas as pd
from PIL import Image
from pathlib import Path
from langchain_core.documents import Document

Why this setup is useful for RAG pipelines?

-->When building a Retrieval-Augmented Generation (RAG) system, it's common to separate different types of extracted content:

#Input PDF: The source document to process.
#extracted_images/: Stores figures, charts, diagrams, logos, and embedded images that can later be analyzed with vision models or OCR.

#page_images/: Stores rendered images of complete pages, which are useful for layout-aware parsing, table detection, or multimodal models.

#parsed_complex_pdf_output/: Serves as the central workspace for all outputs, such as extracted text, tables, metadata, images, and page renders.

In [7]:
# ============================================================
# 1. File Path
# ============================================================

PDF_PATH = Path(r"E:\AIWorkspace\ai-data-engg-ramesh-learnings\Data\complex_rag_parsing_sample_output\complex_rag_parsing_sample_with_image.pdf").resolve()

OUTPUT_DIR = Path(r"E:\AIWorkspace\ai-data-engg-ramesh-learnings\Data\complex_rag_parsing_sample_output\parsed_complex_pdf_output").resolve()

IMAGE_DIR = OUTPUT_DIR / "extracted_images"
PAGE_IMAGE_DIR = OUTPUT_DIR / "page_images"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)
PAGE_IMAGE_DIR.mkdir(parents=True, exist_ok=True)

print("PDF:", PDF_PATH)
print("PDF exists:", PDF_PATH.exists())

PDF: E:\AIWorkspace\ai-data-engg-ramesh-learnings\Data\complex_rag_parsing_sample_output\complex_rag_parsing_sample_with_image.pdf
PDF exists: True


The function's workflow is:

*Open the image.
*Use the Tesseract OCR engine (via pytesseract) to recognize text.
*Clean up the extracted text with .strip().->Remove any extra spaces or blank lines from the beginning and end of the text.
*Return the recognized text (or, if the rest of the function catches *exceptions, return an empty string if OCR cannot be performed).--->Return the extracted text. If the OCR process fails, return an empty string instead.

In [ ]:
# ============================================================
# 2. Helper: Safe OCR
# OCR stands for Optical Character Recognition. In Python, OCR is the process of extracting text from images, scanned documents, screenshots, or PDFs.
# Common OCR Libraries in Python
# - pytesseract (wrapper for Tesseract-OCR)
# - easyocr
# - paddleocr
# ============================================================
def run_ocr_on_image(image_path):
    """
    Runs OCR on image using pytesseract.
    If tesseract is not installed in system, it will return empty text.
    """
    try:
        import pytesseract
        img = Image.open(image_path)
        text = pytesseract.image_to_string(img)
        return text.strip()
    except Exception as e:
        return f"[OCR_SKIPPED_OR_FAILED: {str(e)}]"


*fitz.open(pdf_path) – Opens the PDF file using PyMuPDF.
*len(doc) – Finds the total number of pages in the PDF.
*page.get_text("text") – Extracts selectable text from the current PDF page.
*page.get_images(full=True) – Finds all images present on the current PDF page.
*len(page.get_images(full=True)) – Counts the number of images on the page.
*text.strip() – Removes extra spaces and blank lines from the extracted text.
*page_records.append(page_info) – Adds the extracted page details to the page_records list.

In [9]:
# ============================================================
# 3. Extract Text + Images using PyMuPDF
# ============================================================

def extract_text_and_images(pdf_path):
    doc = fitz.open(pdf_path)

    page_records = []
    image_records = []

    for page_index in range(len(doc)):
        page = doc[page_index]
        page_number = page_index + 1

        # Extract normal selectable text
        text = page.get_text("text")

        # Extract page metadata-like info
        page_info = {
            "page_number": page_number,
            "text": text.strip(),
            "image_count": len(page.get_images(full=True)),
            "width": page.rect.width,
            "height": page.rect.height,
        }

        page_records.append(page_info)

        # Render full page as image for OCR
        pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
        page_image_path = PAGE_IMAGE_DIR / f"page_{page_number:03d}.png"
        pix.save(str(page_image_path))

        # OCR full page image
        ocr_text = run_ocr_on_image(page_image_path)
        page_info["ocr_text"] = ocr_text
        page_info["page_image_path"] = str(page_image_path)

        # Extract embedded images
        images = page.get_images(full=True)

        for img_index, img in enumerate(images):
            xref = img[0]
            base_image = doc.extract_image(xref)

            image_bytes = base_image["image"]
            image_ext = base_image["ext"]

            image_path = IMAGE_DIR / f"page_{page_number:03d}_image_{img_index + 1}.{image_ext}"

            with open(image_path, "wb") as f:
                f.write(image_bytes)

            image_ocr_text = run_ocr_on_image(image_path)

            image_records.append({
                "page_number": page_number,
                "image_index": img_index + 1,
                "image_path": str(image_path),
                "image_ext": image_ext,
                "image_ocr_text": image_ocr_text,
            })

    doc.close()

    return page_records, image_records


In [10]:
page_records, image_records = extract_text_and_images(PDF_PATH)

print("Total pages parsed:", len(page_records))
print("Total images extracted:", len(image_records))

Total pages parsed: 23
Total images extracted: 12


In [11]:
# ============================================================
# 4. Extract Tables using pdfplumber
# ============================================================

def extract_tables(pdf_path):
    table_records = []

    with pdfplumber.open(pdf_path) as pdf:
        for page_index, page in enumerate(pdf.pages):
            page_number = page_index + 1

            try:
                tables = page.extract_tables()
            except Exception as e:
                tables = []
                print(f"Table extraction failed on page {page_number}: {e}")

            for table_index, table in enumerate(tables):
                if not table:
                    continue

                # Clean table rows
                cleaned_table = []
                for row in table:
                    cleaned_row = [
                        cell.strip() if isinstance(cell, str) else cell
                        for cell in row
                    ]
                    cleaned_table.append(cleaned_row)

                # Convert to DataFrame safely
                try:
                    df = pd.DataFrame(cleaned_table[1:], columns=cleaned_table[0])
                except Exception:
                    df = pd.DataFrame(cleaned_table)

                table_records.append({
                    "page_number": page_number,
                    "table_index": table_index + 1,
                    "raw_table": cleaned_table,
                    "markdown": df.to_markdown(index=False),
                    "csv": df.to_csv(index=False)
                })

    return table_records


table_records = extract_tables(PDF_PATH)

print("Total tables extracted:", len(table_records))

for table in table_records[:3]:
    print("\nPage:", table["page_number"], "Table:", table["table_index"])
    print(table["markdown"][:1000])

Total tables extracted: 15

Page: 1 Table: 1
| Section               | Parsing challenge                       | Why it matters for RAG                  |
|:----------------------|:----------------------------------------|:----------------------------------------|
| Contracts             | Dense legal text, clause numbers, cross | Need section-aware chunks and citations |
|                       | references                              |                                         |
| Tables                | Merged headers, numeric columns,        | Need row/column preservation            |
|                       | footnotes                               |                                         |
| Images                | Architecture diagram, heatmap, scanned  | Need OCR or multimodal extraction       |
|                       | form                                    |                                         |
| Multi-tenant metadata | client_id, document_id,                 | Need ac

In [12]:
# ============================================================
# 5. Create LangChain Documents
# ============================================================

langchain_docs = []

# 5.1 Page text documents
for page in page_records:
    page_number = page["page_number"]

    combined_text = f"""
PAGE {page_number}

SELECTABLE TEXT:
{page["text"]}

OCR TEXT:
{page["ocr_text"]}
""".strip()

    doc = Document(
        page_content=combined_text,
        metadata={
            "source": PDF_PATH,
            "page_number": page_number,
            "content_type": "page_text_plus_ocr",
            "image_count": page["image_count"],
            "page_image_path": page["page_image_path"],
        }
    )

    langchain_docs.append(doc)


# 5.2 Table documents
for table in table_records:
    page_number = table["page_number"]

    table_text = f"""
TABLE FOUND ON PAGE {page_number}
TABLE INDEX: {table["table_index"]}

TABLE MARKDOWN:
{table["markdown"]}
""".strip()

    doc = Document(
        page_content=table_text,
        metadata={
            "source": PDF_PATH,
            "page_number": page_number,
            "content_type": "table",
            "table_index": table["table_index"],
        }
    )

    langchain_docs.append(doc)

In [13]:
# 5.3 Image OCR documents

for image in image_records:
    page_number = image["page_number"]

    image_text = f"""
IMAGE FOUND ON PAGE {page_number}
IMAGE INDEX: {image["image_index"]}
IMAGE PATH: {image["image_path"]}

IMAGE OCR TEXT:
{image["image_ocr_text"]}
""".strip()

    doc = Document(
        page_content=image_text,
        metadata={
            "source": PDF_PATH,
            "page_number": page_number,
            "content_type": "image",
            "image_index": image["image_index"],
            "image_path": image["image_path"],
            "image_ext": image["image_ext"],
        }
    )

    langchain_docs.append(doc)


print("Total LangChain Documents created:", len(langchain_docs))

Total LangChain Documents created: 50


In [14]:
# ============================================================
# 6. Save Parsed Output
# ============================================================

# Save page records
with open(OUTPUT_DIR / "page_records.json", "w", encoding="utf-8") as f:
    json.dump(page_records, f, indent=2, ensure_ascii=False)

# Save image records
with open(OUTPUT_DIR / "image_records.json", "w", encoding="utf-8") as f:
    json.dump(image_records, f, indent=2, ensure_ascii=False)

# Save table records
with open(OUTPUT_DIR / "table_records.json", "w", encoding="utf-8") as f:
    json.dump(table_records, f, indent=2, ensure_ascii=False)

# Save all LangChain document content as markdown
with open(OUTPUT_DIR / "rag_ready_documents.md", "w", encoding="utf-8") as f:
    for i, doc in enumerate(langchain_docs):
        f.write(f"\n\n# Document {i + 1}\n")
        f.write(f"\nMetadata:\n```json\n{json.dumps(doc.metadata, indent=2, default=str)}\n```\n")
        f.write("\nContent:\n")
        f.write(doc.page_content)
        f.write("\n\n---\n")

# Save table markdown separately
with open(OUTPUT_DIR / "extracted_tables.md", "w", encoding="utf-8") as f:
    for table in table_records:
        f.write(f"\n\n## Page {table['page_number']} - Table {table['table_index']}\n\n")
        f.write(table["markdown"])
        f.write("\n\n---\n")

print("\nSaved outputs in:", OUTPUT_DIR)



Saved outputs in: E:\AIWorkspace\ai-data-engg-ramesh-learnings\Data\complex_rag_parsing_sample_output\parsed_complex_pdf_output


In [15]:
# ============================================================
# 7. Preview Parsed Output
# ============================================================

if "langchain_docs" not in globals():
    print("langchain_docs is not defined. Run the document-creation cells first.")
    langchain_docs = []

if not langchain_docs:
    print("No LangChain documents are available. Make sure the previous cells have been executed successfully.")
else:
    print("\n================ PAGE TEXT PREVIEW ================\n")
    print(langchain_docs[0].page_content[:1500])

    print("\n================ TABLE PREVIEW ================\n")
    table_docs = [doc for doc in langchain_docs if doc.metadata.get("content_type") == "table"]

    if table_docs:
        print(table_docs[0].page_content[:1500])
    else:
        print("No table docs found.")

    print("\n================ IMAGE OCR PREVIEW ================\n")
    image_docs = [doc for doc in langchain_docs if doc.metadata.get("content_type") == "image"]

    if image_docs:
        print(image_docs[0].page_content[:1500])
    else:
        print("No image docs found.")



================ PAGE TEXT PREVIEW ================

PAGE 1

SELECTABLE TEXT:
Complex RAG Parsing Sample - synthetic document
Page 1
Complex Document for RAG Parsing Tests
Synthetic 15-page PDF with paragraphs, simple and complex tables, diagrams, scanned-form style image, metadata
examples, and production RAG edge cases.
Story Line
Three client teams - Arka Finance, BlueLeaf Retail, and CityRide Mobility - are migrating contracts, policies, support records,
and operational reports into a single RAG platform. Each team has different document types, access rules, and parsing
challenges. The RAG system must answer questions with citations while ensuring that one client never sees another clients
data.
This PDF is intentionally designed to test document loaders, PDF parsers, OCR workflows, table extraction, chunking
strategies, metadata preservation, and source citation quality.
Key statement: RAG does not train the model. RAG gives the model the right context before answering.
Section
P

In [ ]:
Here's a complete walkthrough of your notebook — a **complex PDF parser built for RAG pipelines** that extracts text, tables, images, and OCR content, then packages everything into LangChain `Document` objects. I'll go cell by cell, explaining what it does and *why* that step matters.

## Cell 1 — Title/Header comment
```python
#Complex PDF Parser for RAG
#Text + Tables + Images + OCR + LangChain Documents
```
Just a banner comment. No functional purpose, but it documents intent so anyone reading the notebook immediately knows the scope: this isn't a simple text extractor — it's built to handle **multi-modal PDF content** (text, tables, images, scanned content).

## Cell 2 — Imports
```python
import os, json, fitz, pdfplumber, pandas as pd
from PIL import Image
from pathlib import Path
from langchain_core.documents import Document
```
Each library has a distinct job — this is the core reason the notebook needs *multiple* PDF libraries instead of just one:
- **`fitz` (PyMuPDF)** — fast, reliable extraction of raw selectable text and embedded images, and can render pages as images.
- **`pdfplumber`** — much better at detecting **table structure** (rows/columns) than PyMuPDF. It's slower but geometry-aware.
- **`pandas`** — turns raw table rows into structured `DataFrame`s so tables can be exported as markdown/CSV cleanly.
- **`PIL.Image`** — needed to open extracted images before running OCR.
- **`pathlib.Path`** — cleaner, OS-safe path handling than string concatenation.
- **`langchain_core.documents.Document`** — the standard container LangChain (and most RAG frameworks) expect: `page_content` + `metadata`. Wrapping everything in this format means the output can be dropped directly into a vector store/retriever later.

**Why not just one library?** No single PDF library does text + tables + images + rendering equally well — this is a common real-world pattern: use each tool for its strength.

## Markdown — "Why this setup is useful for RAG"
Explains the folder structure convention: separate raw PDF, embedded images, full-page renders, and final parsed output. This separation matters because **downstream consumers differ** — e.g., a vision model might want `page_images/`, while an OCR pass wants `extracted_images/`, while the retriever wants the final JSON/markdown in `parsed_complex_pdf_output/`.

## Cell 3 — File paths & directory setup
```python
PDF_PATH = Path(...)
OUTPUT_DIR = Path(...)
IMAGE_DIR = OUTPUT_DIR / "extracted_images"
PAGE_IMAGE_DIR = OUTPUT_DIR / "page_images"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
...
```
- Defines where the source PDF lives and where all outputs go.
- `mkdir(parents=True, exist_ok=True)` — creates the directory tree if missing, and **doesn't error out if it already exists** — important for reruns (idempotency).
- Printing `PDF_PATH.exists()` is a sanity check — fail fast if the path is wrong rather than getting a cryptic error deep inside `fitz.open()` later.

## Markdown — OCR function explanation
Describes the logic before showing code (good practice: explain intent, then implementation).

## Cell 4 — `run_ocr_on_image()`
```python
def run_ocr_on_image(image_path):
    try:
        import pytesseract
        img = Image.open(image_path)
        text = pytesseract.image_to_string(img)
        return text.strip()
    except Exception as e:
        return f"[OCR_SKIPPED_OR_FAILED: {str(e)}]"
```
- **Why OCR at all?** Many PDFs contain scanned pages or images with embedded text (forms, diagrams, screenshots) that `fitz`'s `get_text()` can't see, since that only reads the PDF's actual text layer. OCR recovers text from pixels.
- **Why wrapped in try/except instead of failing?** Tesseract is an external system binary, not a pure Python package — it may not be installed on every machine. Rather than crashing the whole pipeline, the function degrades gracefully and tags the failure so you know *why* a document has no OCR text (as you saw in your own output: `tesseract is not installed...`). This is a defensive-coding pattern very important in data pipelines — one missing dependency shouldn't kill the entire run.

## Markdown — PyMuPDF function explanation
Walks through `fitz.open`, `get_text("text")`, `get_images(full=True)` etc. before the code.

## Cell 5 — `extract_text_and_images()`
This is the core PyMuPDF extraction function. Step by step:
1. **`fitz.open(pdf_path)`** — opens the PDF for programmatic access.
2. **Loop over pages** — PDFs are processed page-by-page since RAG chunks are usually page- or section-scoped (keeps citations traceable to a specific page).
3. **`page.get_text("text")`** — pulls selectable/embedded text. This is the fast, accurate path (no OCR error risk) for any born-digital text.
4. **`page_info` dict** — captures structured metadata (page number, dimensions, image count) alongside the text — critical because RAG systems need **metadata for filtering/citation**, not just raw text.
5. **`page.get_pixmap(matrix=fitz.Matrix(2,2))`** — renders the *entire page* as a high-res image (2x zoom for OCR accuracy). This matters because:
   - It lets OCR "see" content that might be visually present but not extractable as text (e.g., diagrams with text baked into shapes, or if the page is a scanned image).
   - It's useful later for multimodal/vision-based retrieval.
6. **OCR on the full page image** — a fallback/supplement to `get_text()`, catching anything missed.
7. **Extract embedded images** (`doc.extract_image(xref)`) — separately pulls out individual images (logos, charts, photos) rather than just the whole-page render, because:
   - Charts/diagrams often carry meaning that pure OCR on a full page would drown out.
   - Each image gets its **own OCR pass** — useful for scanned forms or labeled diagrams.
8. **`doc.close()`** — releases the file handle; good hygiene when processing large PDFs.

**Why separate `page_records` and `image_records`?** Different content types need different downstream handling and different LangChain `Document` metadata (page-level vs. image-level), so keeping them as separate lists simplifies later processing.

## Markdown — table extraction fields
Pre-explains the pdfplumber-based extraction that follows.

## Cell 6 — `extract_tables()`
Uses `pdfplumber` instead of `fitz` specifically because **`fitz` doesn't reliably detect table grid structure** — it just gives flat text — whereas `pdfplumber.extract_tables()` uses line/whitespace geometry to find rows and columns.

Key steps:
1. Loop through pages, call `page.extract_tables()`, wrapped in `try/except` — table detection can throw errors on malformed layouts, and one bad page shouldn't crash the whole document's processing.
2. **Clean each cell** (`cell.strip()`) — raw table cells often have leading/trailing whitespace or newlines from PDF layout quirks.
3. **Convert to `DataFrame`** — treating the first row as header. Wrapped in try/except again because sometimes tables don't have a clean header row (e.g., merged cells) — falls back to a headerless DataFrame rather than crashing.
4. **Store as both markdown and CSV** (`to_markdown()`, `to_csv()`) — markdown is human/LLM-readable (great for RAG context since LLMs parse markdown tables well), while CSV preserves a machine-usable structured form for reprocessing later.

## Cell 7 — Building LangChain `Document`s (Section 5)
This is the **normalization step** — takes three different raw structures (page text+OCR, tables, image OCR) and converts each into the same standard `Document(page_content, metadata)` shape LangChain expects.

- **5.1 Page documents**: combines selectable text + OCR text into one block per page, tagged `content_type: "page_text_plus_ocr"`. Combining both means you don't lose content whichever extraction method worked.
- **5.2 Table documents**: one `Document` per table, with markdown embedded directly in `page_content` so an LLM retrieving this chunk sees a clean table.
- **5.3 Image documents**: one `Document` per extracted image, with OCR text and image path in metadata — lets you later trace a retrieved chunk back to a literal image file if needed (e.g., to re-run OCR or send it to a vision model).

**Why not just dump everything into one big text blob?** Because RAG retrieval quality depends heavily on **chunk granularity and metadata**. Keeping tables, images, and text as separate documents means a retriever can specifically surface a table when the question is about numbers, or an image when the question is about a diagram, rather than pulling in one giant undifferentiated page dump.

## Cell 8 — Save Parsed Output (Section 6)
Persists everything to disk in multiple formats:
- `page_records.json`, `image_records.json`, `table_records.json` — raw structured data, useful for reprocessing/debugging without re-running the whole PDF parse (which is expensive).
- `rag_ready_documents.md` — human-readable dump of every LangChain document + its metadata, handy for manual QA of what the pipeline produced.
- `extracted_tables.md` — tables specifically, since these are often the trickiest content to verify visually.

**Why save at all instead of keeping everything in memory?** PDF parsing (especially OCR + image extraction) is slow. Persisting intermediate outputs lets you skip re-parsing when only downstream steps (like chunking/embedding) change.

## Cell 9 — Preview Parsed Output (Section 7)
A sanity-check/debug cell:
- Guards against running this cell before the pipeline (`if "langchain_docs" not in globals()`) — friendly error instead of a `NameError`.
- Prints a sample page doc, a sample table doc, and a sample image doc so you can visually confirm the pipeline worked as expected before trusting it for a full RAG ingestion run.

---

### Big picture: why this design?
This notebook implements a **defensive, multi-modal PDF-to-RAG pipeline**: it assumes real-world PDFs are messy (mixed scanned/native content, complex tables, embedded diagrams), so it uses the right tool per content type, fails gracefully when a dependency (Tesseract) is missing, and normalizes everything into LangChain's `Document` format with rich metadata — which is exactly what a downstream retriever/vector store needs for accurate, citable answers.

If you want, I can help you fix the OCR issue (Tesseract isn't installed, based on your output), or explain how you'd typically chunk/embed these `Document`s for an actual vector store next.

In [ ]:
PAGE 1

SELECTABLE TEXT:
Complex RAG Parsing Sample - synthetic document
Page 1
Complex Document for RAG Parsing Tests
Synthetic 15-page PDF with paragraphs, simple and complex tables, diagrams, scanned-form style image, metadata
examples, and production RAG edge cases.
Story Line
Three client teams - Arka Finance, BlueLeaf Retail, and CityRide Mobility - are migrating contracts, policies, support records,
and operational reports into a single RAG platform. Each team has different document types, access rules, and parsing
challenges. The RAG system must answer questions with citations while ensuring that one client never sees another clients
data.
This PDF is intentionally designed to test document loaders, PDF parsers, OCR workflows, table extraction, chunking
strategies, metadata preservation, and source citation quality.
Key statement: RAG does not train the model. RAG gives the model the right context before answering.
Section
Parsing challenge
Why it matters for RAG
Contracts
Dense legal text, clause numbers, cross
references
Need section-aware chunks and citations
Tables
Merged headers, numeric columns,
footnotes
Need row/column preservation
Images
Architecture diagram, heatmap, scanned
form
Need OCR or multimodal extraction
Multi-tenant metadata
client_id, document_id,
contract_group_id
Need access-safe retrieval

OCR TEXT:
[OCR_SKIPPED_OR_FAILED: No module named 'pytesseract']

================ TABLE PREVIEW ================

TABLE FOUND ON PAGE 1
TABLE INDEX: 1

TABLE MARKDOWN:
| Section               | Parsing challenge                       | Why it matters for RAG                  |
|:----------------------|:----------------------------------------|:----------------------------------------|
| Contracts             | Dense legal text, clause numbers, cross | Need section-aware chunks and citations |
|                       | references                              |                                         |
| Tables                | Merged headers, numeric columns,        | Need row/column preservation            |
|                       | footnotes                               |                                         |
| Images                | Architecture diagram, heatmap, scanned  | Need OCR or multimodal extraction       |
|                       | form                                    |                                         |
| Multi-tenant metadata | client_id, document_id,                 | Need access-safe retrieval              |
|                       | contract_group_id                       |                                         |

================ IMAGE OCR PREVIEW ================

IMAGE FOUND ON PAGE 2
IMAGE INDEX: 1
IMAGE PATH: D:\MAHA\AIPro\AgenticAI\myAgenticAI_7am_May26\Data\parsed_complex_pdf_output\extracted_images\page_002_image_1.png

IMAGE OCR TEXT:
[OCR_SKIPPED_OR_FAILED: No module named 'pytesseract']
Docling
Docling is an open-source document processing toolkit that converts unstructured documents (such as PDFs, Word files, PowerPoint presentations, HTML pages, images, and spreadsheets) into structured, AI-ready data. It was originally developed by IBM Research, donated to the Linux Foundation, and is now maintained as a community-driven project.

Its main purpose is to prepare documents for applications like:

Retrieval-Augmented Generation (RAG) Knowledge extraction Semantic search AI agents Document question answering

Instead of extracting only plain text, Docling preserves the document's structure, including:

Headings and sections Tables Lists Images and figures Reading order Formulas Metadata Page layout

In [1]:
# # ============================================================
# # Complex PDF Parsing using Docling
# # PDF -> DoclingDocument -> Markdown / JSON / LangChain Documents
# # ============================================================

# # Install dependencies
# !pip install -q docling langchain-core langchain-text-splitters pandas

import json
from pathlib import Path
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

e:\AIWorkspace\ai-data-engg-ramesh-learnings\RagAiAcl\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
PDF_PATH = "E:\AIWorkspace\ai-data-engg-ramesh-learnings\Data\complex_rag_parsing_sample_output\complex_rag_parsing_sample_with_image.pdf")

OUTPUT_DIR = Path("E:\AIWorkspace\ai-data-engg-ramesh-learnings\Data\Docline_parced _output\")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PDF path:", PDF_PATH)

SyntaxError: unmatched ')' (2891876460.py, line 1)